# 🍅 টমেটোর রোগ নির্ণয় — EfficientNetV2-B2 vs MobileNetV3 (Kaggle → server)

**Dataset:** PlantVillage (শুধু টমেটোর ১০টা class) + আপনার `not_leaf` ছবি

**এই notebook কী করে**
1. PlantVillage-এর টমেটো folder খুঁজে app-এর label নামে বদলায় (`Tomato__Target_Spot` → `target_spot`)
2. `not_leaf` class বানায়: আপনার পাতাহীন ছবি + dataset-এর মরিচ ও আলুর পাতা ("এটি টমেটোর পাতা নয়")
3. train/val/test ভাগ করে
4. EfficientNetV2-B2 আর MobileNetV3-Large দুটোই train (label smoothing + MixUp), val দেখে সেরাটা বাছাই
5. Test-এ মূল্যায়ন, আর val থেকে server-এর confidence threshold নিজে ঠিক করে
6. **`tomato_server_bundle.zip`** বানায় → খুললেই `tomato/model.tflite`, `labels.txt`, `meta.json`

> Kaggle Notebook → Settings → **Accelerator: GPU (T4/P100)** আর **Internet: On** করে নিন।
>
> লম্বা training-এ **Save Version → Save & Run All (Commit)** দিন — বিদ্যুৎ/internet চলে গেলেও Kaggle-এর server-এ চলতে থাকবে, ফলাফল Output-এ জমা থাকবে।
>
> ⏩ এখন `RESUME = True`: আগের run-এর `tomato_best_efficientnetv2b2.keras` থেকে training ছাড়াই মূল্যায়ন ও export। নতুন করে train করতে `RESUME = False`।

## ১. Setup ও Config

In [ ]:
import os, random, json, shutil, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ---------------- আপনার dataset path ----------------
BASE = "/kaggle/input/datasets/arjuntejaswi/plant-village/PlantVillage"

# "পাতা নয়" ছবি (মাটি, হাত, আকাশ, অন্য জিনিস) — ভেতরের সব ছবি নেবে
NOT_LEAF_DIR = "/kaggle/input/datasets/jihadulislam1/without-leaves-full"

# Dataset-এর মরিচ ও আলুর পাতাও not_leaf-এ যোগ হবে, যাতে অন্য গাছের পাতা দিলে
# model "টমেটোর পাতা নয়" বলে। প্রতি folder থেকে সর্বোচ্চ এতগুলো ছবি।
OTHER_CROPS_AS_NOT_LEAF = True
OTHER_CROP_PER_FOLDER = 300

# যে class বাদ দিতে চান (app-এর label নামে), যেমন ["target_spot"]
DROP_CLASSES = []

IMG = 288                    # বড় ছবি = ছোট দাগ স্পষ্ট
BATCH = 32
EPOCHS_HEAD = 6             # শুধু নতুন layer train
EPOCHS_FT = 20               # fine-tuning (early stopping আছে)
CANDIDATES = ["efficientnetv2b2", "mobilenetv3large"]   # দুটোই train হবে, val-এ সেরাটা export
LABEL_SMOOTHING = 0.1        # overfitting কমায়
MIXUP_ALPHA = 0.2            # 0 দিলে MixUp বন্ধ

# ---- বিদ্যুৎ/internet চলে গেলে: আগের save করা model থেকে চালু ----
# আগের run-এর tomato_best_<model>.keras (আর থাকলে outputs-এর split_*.csv) Kaggle-এ
# dataset হিসেবে upload করে input-এ যোগ করুন। True হলে training বাদ, সরাসরি মূল্যায়ন ও export।
RESUME = True                # নতুন করে train করতে চাইলে False
# সেই dataset-এর folder; না পেলে পুরো /kaggle/input-এ .keras খুঁজবে
RESUME_DIR = "/kaggle/input/datasets/jihadulislam1/tomato-best-efficientnetv2b2-keras"

RESUME_MODEL = ""
if RESUME:
    if not os.path.isdir(RESUME_DIR):
        print("⚠️ RESUME_DIR পাওয়া যায়নি, পুরো /kaggle/input-এ .keras খুঁজছি...")
        hits = sorted(Path("/kaggle/input").rglob("*.keras"), key=lambda p: "tomato" not in p.name)
        assert hits, "আগের model (.keras) পাওয়া যায়নি — dataset input-এ যোগ করুন, নয়তো RESUME = False"
        RESUME_DIR = str(hits[0].parent)
    keras_files = sorted(Path(RESUME_DIR).rglob("*.keras"))
    assert keras_files, f"{RESUME_DIR}-এ কোনো .keras নেই, আছে: {os.listdir(RESUME_DIR)}"
    RESUME_MODEL = str(keras_files[0])
    print("⏩ আগের model থেকে চালু:", RESUME_MODEL)

WORK = Path("/kaggle/working")
OUT = WORK / "outputs"          # সব ফলাফল এখানে জমা হয়, শেষে এক zip-এ
OUT.mkdir(exist_ok=True)

def savefig(name):
    plt.savefig(OUT / name, dpi=150, bbox_inches="tight")
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

## ২. Dataset খুঁজে বের করা
PlantVillage-এর folder নাম ভিন্ন ভিন্ন বানানে (`Tomato__Tomato_YellowLeaf__Curl_Virus`), তাই নাম ছোট করে মিলিয়ে app-এর label দেওয়া হয়।

In [ ]:
import re

# PlantVillage folder (ছোট হাতের, '_' একটাই) এর শুরু -> app-এর label
# app: lib/data/library/tomato_disease_info.dart, server: models/tomato/labels.txt
FOLDER_TO_LABEL = {
    "tomato_bacterial_spot": "bacterial_spot",
    "tomato_early_blight": "early_blight",
    "tomato_late_blight": "late_blight",
    "tomato_leaf_mold": "leaf_mold",
    "tomato_septoria_leaf_spot": "septoria_leaf_spot",
    "tomato_spider_mites": "spider_mites",
    "tomato_target_spot": "target_spot",
    "tomato_tomato_yellowleaf_curl": "yellow_leaf_curl_virus",
    "tomato_tomato_mosaic": "mosaic_virus",
    "tomato_healthy": "healthy",
}

def norm(name):
    return re.sub(r"_+", "_", name.strip().lower().replace(" ", "_"))

def label_for(folder_name):
    n = norm(folder_name)
    for prefix, label in FOLDER_TO_LABEL.items():
        if n.startswith(prefix):
            return label
    return None

def find_class_root(base):
    best, best_n = None, 0
    for root, dirs, _ in os.walk(base):
        n = sum(label_for(d) is not None for d in dirs)
        if n > best_n:
            best, best_n = root, n
    return best, best_n

search_base = BASE if os.path.exists(BASE) else "/kaggle/input"
if search_base != BASE:
    print("⚠️ BASE path পাওয়া যায়নি, পুরো /kaggle/input-এ খুঁজছি...")
CLASS_ROOT, n_found = find_class_root(search_base)
print("Class root:", CLASS_ROOT, "| টমেটোর class:", n_found)
assert CLASS_ROOT and n_found == 10, "টমেটোর ১০টা folder পাওয়া যায়নি — BASE path দেখুন"
for d in sorted(os.listdir(CLASS_ROOT)):
    print(f"  {d:45s} -> {label_for(d) or ('not_leaf' if norm(d).startswith(('pepper', 'potato')) else '—')}")

## ৩. ছবি সংগ্রহ

In [ ]:
def list_images(folder):
    return [p for p in Path(folder).rglob("*") if p.suffix.lower() in EXTS]

records = []
rng = random.Random(SEED)
for d in sorted(Path(CLASS_ROOT).iterdir()):
    if not d.is_dir():
        continue
    label = label_for(d.name)
    if label in DROP_CLASSES:
        print("বাদ:", d.name); continue
    if label is not None:
        for p in list_images(d):
            records.append((str(p), label))
    elif OTHER_CROPS_AS_NOT_LEAF and norm(d.name).startswith(("pepper", "potato")):
        imgs = list_images(d)
        rng.shuffle(imgs)
        for p in imgs[:OTHER_CROP_PER_FOLDER]:
            records.append((str(p), "not_leaf"))
        print(f"not_leaf-এ যোগ: {d.name} থেকে {min(len(imgs), OTHER_CROP_PER_FOLDER)}টা")

assert os.path.exists(NOT_LEAF_DIR), f"NOT_LEAF_DIR পাওয়া যায়নি: {NOT_LEAF_DIR}"
nl = list_images(NOT_LEAF_DIR)
print("পাতাহীন ছবি:", len(nl))
for p in nl:
    records.append((str(p), "not_leaf"))

df = pd.DataFrame(records, columns=["path", "label"])
print("মোট ছবি (ফিল্টারের আগে):", len(df))

# Dataset পরিষ্কার, তাই নষ্ট/duplicate যাচাই বাদ — সরাসরি training-এ
CLASSES = sorted(df["label"].unique())
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
df["y"] = df["label"].map(CLASS_TO_IDX)
print("\nClasses:", CLASSES)
print(df["label"].value_counts())

In [ ]:
# প্রতি class-এর কয়েকটা ছবি দেখে নিন (ঠিক folder-এ ঠিক রোগ আছে কি না যাচাই)
fig, axes = plt.subplots(len(CLASSES), 4, figsize=(10, 2.4 * len(CLASSES)))
for r, c in enumerate(CLASSES):
    sample = df[df.label == c].sample(4, random_state=1)["path"].tolist()
    for k, p in enumerate(sample):
        axes[r, k].imshow(Image.open(p).convert("RGB"))
        axes[r, k].axis("off")
        if k == 0:
            axes[r, k].set_title(c, fontsize=9, loc="left")
plt.tight_layout(); savefig("01_class_samples.png"); plt.show()

## ৪. Train / Val / Test ভাগ (stratified)
Augmentation শুধু train-এ। ⚠️ PlantVillage-এ একই পাতার একাধিক ছবি আছে, তাই test accuracy বাস্তবের চেয়ে বেশি দেখাবে — নিজের খেতের ছবিতে আলাদা করে পরীক্ষা করবেন।

In [ ]:
# RESUME-এ আগের run-এর split_*.csv পাওয়া গেলে হুবহু সেই ভাগ, নইলে একই seed দিয়ে আবার ভাগ
def prev_split(n):
    return next(Path(RESUME_DIR).rglob(f"split_{n}.csv"), None) if RESUME else None

prev = {n: prev_split(n) for n in ("train", "val", "test")}
if all(prev.values()):
    train_df, val_df, test_df = (pd.read_csv(prev[n]) for n in ("train", "val", "test"))
    for _p in (train_df, val_df, test_df):
        _p["y"] = _p["label"].map(CLASS_TO_IDX)
    assert all(os.path.exists(p) for p in test_df["path"][:20]), "split_*.csv-এর path এই session-এ নেই"
    print("আগের run-এর split_*.csv থেকে একই ভাগ নেওয়া হলো")
else:
    train_df, tmp_df = train_test_split(
        df, test_size=0.30, stratify=df["y"], random_state=SEED)
    val_df, test_df = train_test_split(
        tmp_df, test_size=0.50, stratify=tmp_df["y"], random_state=SEED)

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, len(part))
print(pd.DataFrame({
    "train": train_df.label.value_counts(),
    "val": val_df.label.value_counts(),
    "test": test_df.label.value_counts(),
}))

pd.DataFrame({
    "train": train_df.label.value_counts(),
    "val": val_df.label.value_counts(),
    "test": test_df.label.value_counts(),
}).to_csv(OUT / "class_counts.csv")
for _n, _p in [("train", train_df), ("val", val_df), ("test", test_df)]:
    _p.to_csv(OUT / f"split_{_n}.csv", index=False)

## ৫. tf.data pipeline + augmentation + MixUp
Label এখন one-hot (label smoothing আর MixUp-এর জন্য দরকার)।

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
N_CLASSES = len(CLASSES)

def load_image(path, label):
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, (IMG, IMG))          # 0..255 float
    return tf.cast(img, tf.float32), label

# মাঠের অবস্থা অনুকরণ: ঘোরানো, জুম, আলো-ছায়া, কনট্রাস্ট
augment = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom((-0.15, 0.1)),   # বেশি জুমে দাগ কেটে যায়
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomBrightness(0.3, value_range=(0, 255)),
    layers.RandomContrast(0.3),
], name="augment")

def sample_beta(alpha, n):
    g1 = tf.random.gamma([n], alpha)
    g2 = tf.random.gamma([n], alpha)
    return g1 / (g1 + g2)

# MixUp: দুটো ছবি হালকা মিশিয়ে label-ও মেশায়, model অতি-আত্মবিশ্বাসী হয় না
def mixup(x, y):
    n = tf.shape(x)[0]
    lam = sample_beta(MIXUP_ALPHA, n)
    idx = tf.random.shuffle(tf.range(n))
    lx = tf.reshape(lam, [-1, 1, 1, 1])
    ly = tf.reshape(lam, [-1, 1])
    return x * lx + tf.gather(x, idx) * (1 - lx), y * ly + tf.gather(y, idx) * (1 - ly)

def make_ds(frame, training):
    ds = tf.data.Dataset.from_tensor_slices(
        (frame["path"].values, frame["y"].values.astype("int32")))
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (x, tf.one_hot(y, N_CLASSES)), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH)
    if training:
        ds = ds.map(lambda x, y: (tf.clip_by_value(augment(x, training=True), 0., 255.), y),
                    num_parallel_calls=AUTOTUNE)
        if MIXUP_ALPHA > 0:
            ds = ds.map(mixup, num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds = make_ds(train_df, True)
val_ds = make_ds(val_df, False)
test_ds = make_ds(test_df, False)

# Class imbalance ঠিক করতে weight (one-hot label-এও কাজ করে)
cw = compute_class_weight("balanced", classes=np.arange(N_CLASSES), y=train_df["y"].values)
CLASS_WEIGHT = {i: float(w) for i, w in enumerate(cw)}
print("Class weights:", {CLASSES[i]: round(w, 2) for i, w in CLASS_WEIGHT.items()})

# Augmentation + MixUp কেমন দেখায়
x, y = next(iter(train_ds))
plt.figure(figsize=(10, 3))
for i in range(5):
    plt.subplot(1, 5, i + 1)
    top = int(tf.argmax(y[i]))
    plt.imshow(x[i].numpy().astype("uint8"))
    plt.title(f"{CLASSES[top]} ({float(y[i][top]):.0%})", fontsize=7); plt.axis("off")
savefig("02_augmentation_preview.png"); plt.show()

## ৬. Model গুলো
দুটোতেই `include_preprocessing=True`, তাই **ইনপুট 0–255 float** (app-এ `/255` করা যাবে না)।

In [ ]:
SPECS = {
    "efficientnetv2b2": dict(ctor=keras.applications.EfficientNetV2B2, unfreeze=200, lr_ft=5e-5),
    "mobilenetv3large": dict(ctor=keras.applications.MobileNetV3Large, unfreeze=130, lr_ft=5e-5),
}

def build_model(name):
    base = SPECS[name]["ctor"](
        input_shape=(IMG, IMG, 3), include_top=False, weights="imagenet",
        include_preprocessing=True)
    base.trainable = False
    inp = keras.Input((IMG, IMG, 3), name="image")   # 0..255 float32
    x = base(inp, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(N_CLASSES, activation="softmax", name="probs")(x)
    return keras.Model(inp, out, name=f"tomato_{name}"), base

def compile_model(m, lr):
    m.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
        metrics=[keras.metrics.CategoricalAccuracy(name="accuracy")],
    )

def train_candidate(name):
    spec = SPECS[name]
    m, base = build_model(name)

    # ধাপ ১: শুধু নতুন head
    compile_model(m, 1e-3)
    h1 = m.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD,
               class_weight=CLASS_WEIGHT, callbacks=[
                   keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4,
                                                 restore_best_weights=True),
                   keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                                     patience=2, verbose=1),
               ])

    # ধাপ ২: fine-tuning, BatchNorm frozen
    base.trainable = True
    for layer in base.layers[:-spec["unfreeze"]]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    compile_model(m, spec["lr_ft"])
    ckpt = str(WORK / f"tomato_best_{name}.keras")
    h2 = m.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT,
               class_weight=CLASS_WEIGHT, callbacks=[
                   keras.callbacks.ModelCheckpoint(ckpt, monitor="val_accuracy",
                                                   save_best_only=True, verbose=1),
                   keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5,
                                                 restore_best_weights=True),
                   keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                                     patience=3, verbose=1),
               ])
    return m, h1, h2, ckpt

# TTA: আসল + বাম-ডান উল্টানো + উপর-নিচ উল্টানো, তিনটার গড়
def predict_tta(m, frame):
    ds = make_ds(frame, False)
    p = m.predict(ds, verbose=0)
    p += m.predict(ds.map(lambda x, y: (tf.image.flip_left_right(x), y)), verbose=0)
    p += m.predict(ds.map(lambda x, y: (tf.image.flip_up_down(x), y)), verbose=0)
    return p / 3

## ৭. Training — প্রতিটা model
ছবি অনেক বেশি, তাই GPU-তে মোট ~১.৫–২.৫ ঘণ্টা লাগতে পারে। শুধু একটা চাইলে `CANDIDATES`-এ একটাই রাখুন।

In [ ]:
def load_saved(path):
    # আগের run-এর ModelCheckpoint (.keras); না খুললে একই model বানিয়ে শুধু weights বসানো
    stem = Path(path).stem.lower()
    try:
        m = keras.models.load_model(path, compile=False)
    except Exception as e:
        print("load_model হয়নি, weights আলাদা করে বসাচ্ছি:", e)
        m, _ = build_model(next(k for k in SPECS if k in stem))
        m.load_weights(path)
    name = next((k for k in SPECS if k in f"{stem} {m.name}".lower()), stem)
    assert m.output_shape[-1] == N_CLASSES, \
        f"model-এ {m.output_shape[-1]}টা class, data-তে {N_CLASSES}টা — dataset/DROP_CLASSES আগের মতো রাখুন"
    return name, m

results = {}
y_val = val_df["y"].values

def evaluate_candidate(name, m, h1_, h2_, ckpt):
    val_plain = float((m.predict(val_ds, verbose=0).argmax(1) == y_val).mean())
    val_tta = float((predict_tta(m, val_df).argmax(1) == y_val).mean())
    results[name] = dict(model=m, h1=h1_, h2=h2_, ckpt=ckpt,
                         val_plain=val_plain, val_tta=val_tta)
    print(f"{name}: val {val_plain*100:.2f}% | val+TTA {val_tta*100:.2f}%")

if RESUME:
    name, m = load_saved(RESUME_MODEL)
    print("⏩ training বাদ — আগের model:", name)
    evaluate_candidate(name, m, None, None, RESUME_MODEL)
else:
    for name in CANDIDATES:
        print("\n" + "=" * 25, name, "=" * 25)
        evaluate_candidate(name, *train_candidate(name))

## ৮. তুলনা ও সেরা model বাছাই
বাছাই **val** দিয়ে। Test শুধু দেখানোর জন্য, তাই test সংখ্যা সৎ থাকে।

In [ ]:
y_test = test_df["y"].values
rows = []
for name, r in results.items():
    t_plain = float((r["model"].predict(test_ds, verbose=0).argmax(1) == y_test).mean())
    t_tta = float((predict_tta(r["model"], test_df).argmax(1) == y_test).mean())
    r["test_plain"], r["test_tta"] = t_plain, t_tta
    rows.append({"model": name, "params_M": round(r["model"].count_params() / 1e6, 2),
                 "val %": round(r["val_plain"] * 100, 2), "val+TTA %": round(r["val_tta"] * 100, 2),
                 "test %": round(t_plain * 100, 2), "test+TTA %": round(t_tta * 100, 2)})
comparison = pd.DataFrame(rows)
comparison.to_csv(OUT / "model_comparison.csv", index=False)
print(comparison.to_string(index=False))

BEST_NAME = max(results, key=lambda n: results[n]["val_plain"])
best = results[BEST_NAME]
model, h1, h2, BEST = best["model"], best["h1"], best["h2"], best["ckpt"]
# server প্রতি ছবি একবারই চালায়, তাই TTA শুধু তুলনার জন্য মাপা, export-এ বন্ধ
USE_TTA_FINAL = False
print(f"\nসেরা model: {BEST_NAME} | TTA: {'চালু' if USE_TTA_FINAL else 'বন্ধ'}")

In [ ]:
if h1 is None:
    print("আগের save করা model — training history নেই, তাই chart বাদ")
else:
    acc = h1.history["accuracy"] + h2.history["accuracy"]
    val_acc = h1.history["val_accuracy"] + h2.history["val_accuracy"]
    loss = h1.history["loss"] + h2.history["loss"]
    val_loss = h1.history["val_loss"] + h2.history["val_loss"]
    split = len(h1.history["accuracy"])

    pd.DataFrame({"accuracy": acc, "val_accuracy": val_acc, "loss": loss, "val_loss": val_loss}).to_csv(
        OUT / "training_history.csv", index_label="epoch")

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(acc, label="train"); ax[0].plot(val_acc, label="val")
    ax[0].axvline(split - 1, ls="--", c="gray"); ax[0].set_title(f"Accuracy ({BEST_NAME})"); ax[0].legend()
    ax[1].plot(loss, label="train"); ax[1].plot(val_loss, label="val")
    ax[1].axvline(split - 1, ls="--", c="gray"); ax[1].set_title("Loss"); ax[1].legend()
    savefig("03_training_curves.png"); plt.show()

## ৯. Test set-এ সৎ মূল্যায়ন
Test ছবি training-এ কখনো দেখেনি। **এই সংখ্যাটাই আসল।**

In [ ]:
y_true = test_df["y"].values
probs_plain = model.predict(test_ds, verbose=0)
probs = predict_tta(model, test_df) if USE_TTA_FINAL else probs_plain
print("Model:", BEST_NAME, "| TTA:", USE_TTA_FINAL)
y_pred = probs.argmax(1)
conf = probs.max(1)

print("Test accuracy:", round((y_pred == y_true).mean() * 100, 2), "%\n")
report_txt = classification_report(y_true, y_pred, target_names=CLASSES, digits=3)
print(report_txt)
TEST_ACC = float((y_pred == y_true).mean())
(OUT / "classification_report.txt").write_text(
    f"Test accuracy: {TEST_ACC*100:.2f}%" + chr(10) * 2 + report_txt, encoding="utf-8")
json.dump(classification_report(y_true, y_pred, target_names=CLASSES, output_dict=True),
          open(OUT / "classification_report.json", "w"), indent=2)
pd.DataFrame({"path": test_df["path"].values,
              "actual": [CLASSES[i] for i in y_true],
              "predicted": [CLASSES[i] for i in y_pred],
              "confidence": conf}).to_csv(OUT / "test_predictions.csv", index=False)

cm = confusion_matrix(y_true, y_pred)
pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(OUT / "confusion_matrix.csv")
fig, ax = plt.subplots(figsize=(10, 9))
ax.imshow(cm, cmap="Greens")
ax.set_xticks(range(len(CLASSES))); ax.set_yticks(range(len(CLASSES)))
ax.set_xticklabels(CLASSES, rotation=45, ha="right"); ax.set_yticklabels(CLASSES)
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); plt.tight_layout()
savefig("04_confusion_matrix.png"); plt.show()

In [ ]:
# Confidence threshold ঠিক করা: threshold-এর ওপরে কতগুলো ছবিতে মত দেয় আর সেগুলো কত % ঠিক
thr_rows = []
print(f"{'threshold':>10} {'মত দেয় %':>10} {'সঠিক %':>10}")
for t in [0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    m = conf >= t
    covered = m.mean() * 100
    correct = (y_pred[m] == y_true[m]).mean() * 100 if m.any() else float("nan")
    print(f"{t:>10} {covered:>10.1f} {correct:>10.1f}")
    thr_rows.append({"threshold": t, "covered_percent": round(covered, 1), "correct_percent": round(correct, 1)})
pd.DataFrame(thr_rows).to_csv(OUT / "confidence_thresholds.csv", index=False)

# server-এর threshold val দিয়ে ঠিক করা (test দিয়ে নয়, যাতে test সৎ থাকে):
# ০.৬ থেকে শুরু করে সবচেয়ে ছোট t যেখানে t-এর ওপরের ছবিগুলোর ≥৯৮% সঠিক।
# ০.৬-এর নিচে নয়: খেতের ছবি dataset-এর ছবির চেয়ে কঠিন, val-এ নিচু t ভালো দেখালেও মাঠে ঝুঁকি
val_probs = model.predict(val_ds, verbose=0)
v_pred, v_conf = val_probs.argmax(1), val_probs.max(1)
CONFIDENT_T = 0.6
for t in np.arange(0.60, 0.91, 0.05):
    mask = v_conf >= t
    if mask.any() and (v_pred[mask] == y_val[mask]).mean() >= 0.98:
        CONFIDENT_T = round(float(t), 2)
        break
MIN_T = round(max(0.40, CONFIDENT_T - 0.2), 2)
print(f"\nServer threshold → নিশ্চিত: ≥{CONFIDENT_T} | সম্ভাব্য: ≥{MIN_T}")

In [ ]:
# মাঠের কঠিন অবস্থার পরীক্ষা: অন্ধকার/ঝাপসা/noise যোগ করে test ছবি আবার চালানো
def degrade(x, y):
    x = tf.image.random_brightness(x, 50.0)
    x = tf.image.random_contrast(x, 0.6, 1.4)
    x = x + tf.random.normal(tf.shape(x), stddev=12.0)
    return tf.clip_by_value(x, 0., 255.), y

hard_ds = make_ds(test_df, False).map(degrade)
hard_probs = model.predict(hard_ds, verbose=0)
hard_acc = (hard_probs.argmax(1) == y_true).mean() * 100
print(f"পরিষ্কার test: {(y_pred == y_true).mean()*100:.1f}%  |  খারাপ অবস্থার test: {hard_acc:.1f}%")
HARD_ACC = float(hard_acc) / 100

## ১০. কোথায় ভুল করছে দেখুন

In [ ]:
wrong = np.where(y_pred != y_true)[0]
pd.DataFrame({"path": test_df.iloc[wrong]["path"].values,
              "actual": [CLASSES[i] for i in y_true[wrong]],
              "predicted": [CLASSES[i] for i in y_pred[wrong]],
              "confidence": conf[wrong]}).to_csv(OUT / "wrong_predictions.csv", index=False)
print("ভুল:", len(wrong), "/", len(y_true))
show = wrong[:12]
if len(show):
    cols = 4
    rows = int(np.ceil(len(show) / cols))
    plt.figure(figsize=(12, 3.2 * rows))
    for k, i in enumerate(show):
        plt.subplot(rows, cols, k + 1)
        plt.imshow(Image.open(test_df.iloc[i]["path"]).convert("RGB"))
        plt.title(f"সঠিক: {CLASSES[y_true[i]]}\nবলেছে: {CLASSES[y_pred[i]]} ({conf[i]:.0%})", fontsize=8)
        plt.axis("off")
    plt.tight_layout(); savefig("05_wrong_predictions.png"); plt.show()

## ১১. TFLite-এ রূপান্তর (float16, ছোট size)

In [ ]:
SAVED = str(WORK / "saved_model")
if os.path.exists(SAVED):
    shutil.rmtree(SAVED)
try:
    model.export(SAVED)                       # Keras 3
except AttributeError:
    model.save(SAVED, save_format="tf")       # Keras 2

converter = tf.lite.TFLiteConverter.from_saved_model(SAVED)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_bytes = converter.convert()

SERVER_DIR = WORK / "tomato"            # server-এ ml_backend/models/tomato/ হবে
SERVER_DIR.mkdir(exist_ok=True)
TFLITE_PATH = SERVER_DIR / "model.tflite"
TFLITE_PATH.write_bytes(tflite_bytes)
print(f"tomato/model.tflite: {len(tflite_bytes)/1e6:.2f} MB")

In [ ]:
# TFLite model Keras-এর মতোই ঠিক চলছে কি না যাচাই (server-এ যা চলবে সেটাই)
interp = tf.lite.Interpreter(model_path=str(TFLITE_PATH))
interp.allocate_tensors()
inp_d = interp.get_input_details()[0]
out_d = interp.get_output_details()[0]
print("Input :", inp_d["shape"], inp_d["dtype"])
print("Output:", out_d["shape"], out_d["dtype"])

correct = 0
for p, y in zip(test_df["path"], test_df["y"]):
    img, _ = load_image(p, 0)
    interp.set_tensor(inp_d["index"], img.numpy()[None, ...].astype(np.float32))
    interp.invoke()
    pred = int(np.argmax(interp.get_tensor(out_d["index"])[0]))
    correct += int(pred == y)
TFLITE_ACC = correct / len(test_df)
print(f"TFLite test accuracy: {correct/len(test_df)*100:.2f}%  (Keras, TTA ছাড়া: {(probs_plain.argmax(1)==y_true).mean()*100:.2f}%)")

## ১২. Server-এর জন্য file তৈরি

In [ ]:
# app-এর tomato_disease_info.dart-এর নামের সাথে মিল রাখা
BN = {
    "bacterial_spot": "ব্যাকটেরিয়াজনিত দাগ রোগ",
    "early_blight": "আগাম ধসা রোগ",
    "late_blight": "নাবী ধসা রোগ",
    "leaf_mold": "পাতার ছাঁচ রোগ",
    "septoria_leaf_spot": "সেপ্টোরিয়া পাতার দাগ",
    "spider_mites": "লাল মাকড়",
    "target_spot": "টার্গেট স্পট",
    "yellow_leaf_curl_virus": "পাতা কোঁকড়ানো ভাইরাস",
    "mosaic_virus": "মোজাইক ভাইরাস",
    "healthy": "সুস্থ টমেটো গাছ",
    "not_leaf": "টমেটোর পাতা নয়",
}

(SERVER_DIR / "labels.txt").write_text("\n".join(CLASSES), encoding="utf-8")

meta = {
    "model": BEST_NAME,
    "model_version": f"tomato-{BEST_NAME}",
    "input_size": IMG,
    "input_dtype": "float32",
    "input_range": "0-255 (scale ভেতরে করা আছে)",
    "color_order": "RGB",
    "classes": CLASSES,
    "classes_bn": [BN.get(c, c) for c in CLASSES],
    "confident_threshold": CONFIDENT_T,
    "min_threshold": MIN_T,
    "test_accuracy": round(float((y_pred == y_true).mean()), 4),
}
(SERVER_DIR / "meta.json").write_text(
    json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")

# খুললেই tomato/ folder — সরাসরি ml_backend/models/-এ রাখুন
with zipfile.ZipFile(WORK / "tomato_server_bundle.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in ["model.tflite", "labels.txt", "meta.json"]:
        z.write(SERVER_DIR / f, f"tomato/{f}")

# ---------- সব ফলাফল এক zip-এ ----------
metrics = {
    "best_model": BEST_NAME,
    "tta": bool(USE_TTA_FINAL),
    "keras_test_accuracy": round(TEST_ACC, 4),
    "comparison": comparison.to_dict(orient="records"),
    "tflite_test_accuracy": round(float(TFLITE_ACC), 4),
    "hard_conditions_test_accuracy": round(HARD_ACC, 4),
    "n_train": len(train_df), "n_val": len(val_df), "n_test": len(test_df),
    "classes": CLASSES,
}
(OUT / "metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
(OUT / "config.json").write_text(json.dumps({
    "IMG": IMG, "BATCH": BATCH, "EPOCHS_HEAD": EPOCHS_HEAD, "EPOCHS_FT": EPOCHS_FT,
    "CANDIDATES": CANDIDATES, "LABEL_SMOOTHING": LABEL_SMOOTHING, "MIXUP_ALPHA": MIXUP_ALPHA,
    "SPECS": {k: {"unfreeze": v["unfreeze"], "lr_ft": v["lr_ft"]} for k, v in SPECS.items()}, "DROP_CLASSES": DROP_CLASSES, "SEED": SEED,
    "NOT_LEAF_DIR": NOT_LEAF_DIR, "OTHER_CROPS_AS_NOT_LEAF": OTHER_CROPS_AS_NOT_LEAF,
    "OTHER_CROP_PER_FOLDER": OTHER_CROP_PER_FOLDER, "BASE": BASE,
    "CONFIDENT_T": CONFIDENT_T, "MIN_T": MIN_T,
    "RESUME": RESUME, "RESUME_MODEL": RESUME_MODEL,
}, ensure_ascii=False, indent=2), encoding="utf-8")
shutil.copytree(SERVER_DIR, OUT / "tomato", dirs_exist_ok=True)
if os.path.exists(BEST):
    shutil.copy(BEST, OUT / "tomato_best.keras")

ALL_ZIP = WORK / "tomato_all_outputs.zip"
with zipfile.ZipFile(ALL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*")):
        if p.is_file():
            z.write(p, p.relative_to(OUT))
print("tomato_all_outputs.zip এর ভেতরে:")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(OUT), round(p.stat().st_size / 1e6, 3), "MB")
print()

print(open(SERVER_DIR / "labels.txt", encoding="utf-8").read())
print()
for f in ["tomato/model.tflite", "tomato/labels.txt", "tomato/meta.json",
          "tomato_server_bundle.zip", "tomato_all_outputs.zip"]:
    print(f, round((WORK / f).stat().st_size / 1e6, 3), "MB")

## ✅ শেষ ধাপ
Kaggle-এর **Output** ট্যাব থেকে দুটো zip download করুন:
- **`tomato_server_bundle.zip`** — খুলে ভেতরের `tomato/` folder-টা project-এর **`ml_backend/models/`**-এ রাখুন। এতে আছে `model.tflite`, `labels.txt`, `meta.json` (threshold সহ)।
- **`tomato_all_outputs.zip`** — সব chart, report, confusion matrix, ভুল prediction, আর `tomato_best.keras`।

তারপর server চালিয়ে পরীক্ষা করুন:
```
python -m ml_backend.api.main
curl -F "crop=tomato" -F "file=@leaf.jpg" http://localhost:8000/predict
```
`/health`-এ `"tomato": true` দেখালে app-এ টমেটো বাছলে "সার্ভার চালু" দেখাবে।

**মনে রাখবেন**
- PlantVillage ল্যাবের ছবি, তাই test accuracy বেশি আসবে। নিজের খেতের টমেটো পাতার ছবি দিয়ে আসল পরীক্ষা করুন।
- Early blight ↔ Target spot ↔ Septoria দেখতে কাছাকাছি; confusion matrix-এ এদের দিকে নজর দিন।